# Tercera entrega — Modelo físico, rutinas e interfaz gráfica
### Gestión de clientes y operaciones bancarias (dataset Berka)
**Juan Felipe Molano Vega · Juan Pablo Castaño Morales**

Este notebook implementa, paso a paso, lo solicitado en la tercera entrega:

| Requisito | Dónde se resuelve |
|---|---|
| Correcciones de la segunda entrega | [Sección 2](#2) |
| Modelo físico implementado en el motor de BD (MySQL) | [Secciones 3, 4 y 5](#3) |
| Interfaz de **al menos 2 tablas** (inserción, borrado, actualización y consulta) | [Sección 9](#9): tablas `Cliente` y `Transacciones` |
| **2 triggers** | [Sección 8](#8): `trg_transacciones_before_insert`, `trg_cliente_after_update` |
| **1 procedimiento** | [Sección 7](#7): `sp_extracto_cuenta` |
| **1 función** | [Sección 6](#6): `fn_saldo_cuenta` |

**Índice**
1. Configuración y conexión
2. Correcciones de la segunda entrega
3. Carga del dataset original (`berka.sql`) en una base de staging
4. Modelo físico: creación de la base `banco_berka`
5. Migración de los datos al modelo y validación
6. Función `fn_saldo_cuenta`
7. Procedimiento `sp_extracto_cuenta`
8. Triggers
9. Interfaz gráfica
10. Resumen de objetos creados

> **Requisitos:** MySQL 8.0.16 o superior (para que se apliquen las restricciones `CHECK`) y el entorno del proyecto instalado con `uv sync` (ver `README.md`). El dataset se lee de `data/berka.sql.gz`. Las celdas se ejecutan en orden, de arriba hacia abajo.

<a id="1"></a>
## 1. Configuración y conexión

### 1.1 Librerías
- `pymysql`: driver de Python para MySQL.
- `sqlalchemy`: manejo de la conexión y consultas parametrizadas.
- `pandas`: mostrar resultados como tablas.
- `ipywidgets`: construir la interfaz gráfica dentro del notebook.
- `cryptography`: requerido por `pymysql` para la autenticación por defecto de MySQL 8.

Las librerías se instalan con `uv sync` desde la raíz del proyecto; este notebook usa el kernel del entorno `.venv`.

### 1.2 Parámetros de conexión
Los parámetros se leen de `config/config.toml` (copia de `config/config.example.toml`): host, puerto, usuario, contraseña, nombres de las bases y ruta del dataset. Si la contraseña está vacía en el archivo, se toma de la variable de entorno `MYSQL_PASSWORD` o se pide al ejecutar la celda.

In [ ]:
import time
from datetime import date
import pandas as pd
import pymysql
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

from berka_bank.config import load_settings

CFG = load_settings()  # config/config.toml

MYSQL_HOST = CFG.host
MYSQL_PORT = CFG.port
MYSQL_USER = CFG.user
MYSQL_PASSWORD = CFG.password

DB_ORIGEN = CFG.staging_db        # base que crea berka.sql (datos crudos, "staging")
DB_MODELO = CFG.model_db          # base del modelo físico de esta entrega
RUTA_BERKA_SQL = CFG.berka_sql    # script del dataset (data/berka.sql.gz)

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)


### 1.3 Funciones de apoyo
- `crear_engine(db)`: crea la conexión de SQLAlchemy (a una base concreta o solo al servidor).
- `ejecutar(sql, params)`: ejecuta una sentencia dentro de una transacción (hace *commit* al final o *rollback* si falla).
- `consultar(sql, params)`: devuelve el resultado de un `SELECT` como `DataFrame`.

In [2]:
def crear_engine(db=None):
    url = URL.create('mysql+pymysql', username=MYSQL_USER, password=MYSQL_PASSWORD,
                     host=MYSQL_HOST, port=MYSQL_PORT, database=db,
                     query={'charset': 'utf8mb4'})
    return create_engine(url, pool_pre_ping=True)

engine_srv = crear_engine()   # conexión al servidor, sin base seleccionada
engine = None                 # conexión a banco_berka (se crea en la sección 4)

def ejecutar(sql, params=None, eng=None):
    with (eng or engine).begin() as con:
        return con.execute(text(sql), params or {})

def consultar(sql, params=None, eng=None):
    with (eng or engine).connect() as con:
        return pd.read_sql(text(sql), con, params=params or {})

with engine_srv.connect() as con:
    print('Conectado a MySQL', con.execute(text('SELECT VERSION()')).scalar())

OperationalError: (pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on 'localhost' ([WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión)")
(Background on this error at: https://sqlalche.me/e/21/e3q8)

<a id="2"></a>
## 2. Correcciones de la segunda entrega

Al contrastar el modelo lógico (`sql_Entrega2.mwb`) con los datos, el diccionario de datos dejó registradas varias observaciones (sección 3 del documento). En el modelo físico se corrigen así:

| # | Observación de la segunda entrega | Corrección en el modelo físico |
|---|---|---|
| 1 | Índices `UNIQUE` sobre llaves foráneas (`Transacciones.Identificador_Cuenta`, `Disposicion.Identificador_Cuenta`, `Prestamos.Codigo_Estado`, `Cliente.ID_Distrito`, etc.) impedían cargar los datos porque esos valores se repiten. | Se eliminan. Solo son únicas las llaves primarias, `Distrito.Nombre_Distrito`, `Tarjeta.Codigo_Disposicion` (una tarjeta como máximo por disposición) y `Prestamos.Identificador_Cuenta` (un préstamo como máximo por cuenta). |
| 2 | `Cuenta.Codigo_Prestamos` era obligatorio, pero solo 682 de 4,500 cuentas tienen préstamo (relación 0..1). | La llave se mueve a `Prestamos.Identificador_Cuenta` (`NOT NULL`, `UNIQUE`, FK → `Cuenta`). Una cuenta sin préstamo simplemente no aparece en `Prestamos`. |
| 3 | `Prestamos.Duracion_Prestamo` estaba como fecha/hora, pero los datos son meses. | `INT` con `CHECK (Duracion_Prestamo IN (12,24,36,48,60))`. |
| 4 | `Estado_Prestamo.Codigo_Estado` era entero, pero en los datos el estado es una letra. | Se cambia a `CHAR(1)` con los valores originales `A`, `B`, `C`, `D`. Así no hace falta una tabla de equivalencias. |
| 5 | `Transacciones.k_symbol` no tenía campo en el modelo (trae información en 521,006 filas). | Nueva columna `Transacciones.Caracterizacion_Operacion`. |
| 6 | Montos y tasas como `FLOAT`. | `DECIMAL(12,2)` para dinero y `DECIMAL(5,2)` para porcentajes. |
| 7 | Columna `Nombre_Districto` mal escrita e índice `Nombre_Distrito_UNIQUE` redundante con la PK. | Se renombra a `Nombre_Distrito` (con `UNIQUE` sobre el nombre) y se elimina el índice redundante. |
| 8 | Varias FK formaban parte de la PK (relaciones identificadoras, marcadas "PK, FK" en el diccionario). | Cada tabla tiene una PK simple (su identificador). Las FK quedan como columnas normales `NOT NULL` con índice. |
| 9 | Fechas en formato `AAMMDD` y género codificado dentro de `birth_number`. | La migración convierte a `DATE` y deriva `Genero` (`M`/`F`) del mes (mes > 50 = mujer). |

Además se añaden:
- Restricciones `CHECK` para los dominios del diccionario (tipos de tarjeta, tipos de movimiento, frecuencias, género, montos ≥ 0, etc.).
- `AUTO_INCREMENT` en las PK, para que la interfaz pueda insertar registros nuevos sin calcular el ID.
- Un índice `(Identificador_Cuenta, Fecha_Realizada, Codigo_Transacciones)` en `Transacciones`, que usan la función, el procedimiento y la interfaz para consultar por cuenta y fecha.
- La tabla `Auditoria_Cliente`, que llena uno de los triggers.

<a id="3"></a>
## 3. Carga del dataset original en la base de staging (`berka`)

`berka.sql` crea la base `berka` con las 8 tablas originales (`account`, `card`, `client`, `disp`, `district`, `loan`, `order`, `trans`) y más de un millón de filas. Se carga tal cual como **staging**. Desde ahí se migra al modelo corregido en la sección 5.

El script se lee línea por línea y se ejecuta sentencia por sentencia (cada `INSERT` trae unas 1,000 filas), así no se supera el límite `max_allowed_packet` del servidor. Si la base ya está cargada, este paso se omite. Para forzar la recarga, pon `RECARGAR_STAGING = True`.

*Tiempo aproximado: 1 a 3 minutos.*

In [3]:
RECARGAR_STAGING = False

# Lee el script sentencia por sentencia (acepta .sql y .sql.gz)
from berka_bank.staging import read_statements as leer_sentencias

def staging_cargado():
    try:
        n = consultar(f'SELECT COUNT(*) AS n FROM {DB_ORIGEN}.trans', eng=engine_srv)['n'][0]
        return n == 1_056_320
    except Exception:
        return False

if staging_cargado() and not RECARGAR_STAGING:
    print(f"La base '{DB_ORIGEN}' ya está cargada; se omite este paso.")
else:
    inicio, n = time.time(), 0
    con = pymysql.connect(host=MYSQL_HOST, port=MYSQL_PORT, user=MYSQL_USER,
                          password=MYSQL_PASSWORD, charset='utf8mb4')
    try:
        with con.cursor() as cur:
            for sentencia in leer_sentencias(RUTA_BERKA_SQL):
                cur.execute(sentencia)
                n += 1
                if n % 50 == 0:
                    print(f'\r{n} sentencias ejecutadas...', end='')
        con.commit()
    finally:
        con.close()
    print(f'\rListo: {n} sentencias en {time.time() - inicio:.0f} s.        ')

OperationalError: (2003, "Can't connect to MySQL server on 'localhost' ([WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión)")

### 3.1 Verificación del staging

In [ ]:
tablas_origen = ['district', 'client', 'account', 'disp', 'card', 'loan', 'order', 'trans']
pd.DataFrame([
    {'tabla': t, 'filas': consultar(f'SELECT COUNT(*) AS n FROM {DB_ORIGEN}.`{t}`', eng=engine_srv)['n'][0]}
    for t in tablas_origen
])

<a id="4"></a>
## 4. Modelo físico: creación de la base `banco_berka`

Se crean las 9 tablas del modelo (más la tabla de auditoría) con las correcciones de la sección 2. El orden respeta las dependencias: primero las tablas sin llaves foráneas (`Distrito`, `Estado_Prestamo`) y después las que dependen de ellas.

```
Distrito ─┬─< Cliente ──< Disposicion >── Cuenta >── Distrito
          └─< Cuenta ─┬─< Disposicion ──o Tarjeta
                      ├─< Transacciones
                      ├─< Ordenes_de_pago
                      └─o Prestamos >── Estado_Prestamo
```
(`─<` uno a muchos, `─o` uno a cero-o-uno)

> ⚠️ La celda **borra y vuelve a crear** `banco_berka`, así que se puede volver a ejecutar desde cero.

In [4]:
DDL_MODELO = f"""
DROP DATABASE IF EXISTS {DB_MODELO};
CREATE DATABASE {DB_MODELO} CHARACTER SET utf8mb4 COLLATE utf8mb4_0900_ai_ci;
USE {DB_MODELO};

-- 2.1 Distrito
CREATE TABLE Distrito (
    ID_Distrito             INT           NOT NULL,
    Nombre_Distrito         VARCHAR(80)   NOT NULL,
    Region_que_pertenece    VARCHAR(30)   NOT NULL,
    Cantidad_Municipios     INT           NOT NULL,
    Tasa_Desempleo          DECIMAL(5,2)  NOT NULL,
    Salario_Promedio        DECIMAL(10,2) NOT NULL,
    Cantidad_Emprendedores  INT           NOT NULL,
    Grado_Urbanizacion      DECIMAL(5,2)  NOT NULL,
    Numero_Habitantes       INT           NOT NULL,
    CONSTRAINT pk_distrito PRIMARY KEY (ID_Distrito),
    CONSTRAINT uq_distrito_nombre UNIQUE (Nombre_Distrito),
    CONSTRAINT ck_distrito_municipios CHECK (Cantidad_Municipios >= 1),
    CONSTRAINT ck_distrito_desempleo  CHECK (Tasa_Desempleo BETWEEN 0 AND 100),
    CONSTRAINT ck_distrito_urbaniz    CHECK (Grado_Urbanizacion BETWEEN 0 AND 100),
    CONSTRAINT ck_distrito_habitantes CHECK (Numero_Habitantes > 0)
);

-- 2.2 Cliente
CREATE TABLE Cliente (
    Identificador_Cliente  INT      NOT NULL AUTO_INCREMENT,
    Fecha_Nacimiento       DATE     NOT NULL,
    Genero                 CHAR(1)  NOT NULL,
    ID_Distrito            INT      NOT NULL,
    CONSTRAINT pk_cliente PRIMARY KEY (Identificador_Cliente),
    CONSTRAINT fk_cliente_distrito FOREIGN KEY (ID_Distrito) REFERENCES Distrito (ID_Distrito),
    CONSTRAINT ck_cliente_genero CHECK (Genero IN ('M', 'F')),
    CONSTRAINT ck_cliente_fecha  CHECK (Fecha_Nacimiento >= '1900-01-01')
);

-- 2.3 Cuenta (ya no tiene Codigo_Prestamos: la relación se guarda en Prestamos)
CREATE TABLE Cuenta (
    Identificador_Cuenta              INT          NOT NULL AUTO_INCREMENT,
    Fecha_Apertura                    DATE         NOT NULL,
    Frecuencia_Emision_Estado_Cuenta  VARCHAR(30)  NOT NULL,
    ID_Distrito                       INT          NOT NULL,
    CONSTRAINT pk_cuenta PRIMARY KEY (Identificador_Cuenta),
    CONSTRAINT fk_cuenta_distrito FOREIGN KEY (ID_Distrito) REFERENCES Distrito (ID_Distrito),
    CONSTRAINT ck_cuenta_frecuencia CHECK (Frecuencia_Emision_Estado_Cuenta IN
        ('POPLATEK MESICNE', 'POPLATEK TYDNE', 'POPLATEK PO OBRATU'))
);

-- 2.4 Disposicion
CREATE TABLE Disposicion (
    Codigo_Disposicion     INT          NOT NULL AUTO_INCREMENT,
    Tipo_Disposicion       VARCHAR(45)  NOT NULL,
    Identificador_Cliente  INT          NOT NULL,
    Identificador_Cuenta   INT          NOT NULL,
    CONSTRAINT pk_disposicion PRIMARY KEY (Codigo_Disposicion),
    CONSTRAINT fk_disp_cliente FOREIGN KEY (Identificador_Cliente) REFERENCES Cliente (Identificador_Cliente),
    CONSTRAINT fk_disp_cuenta  FOREIGN KEY (Identificador_Cuenta)  REFERENCES Cuenta (Identificador_Cuenta),
    CONSTRAINT ck_disp_tipo CHECK (Tipo_Disposicion IN ('OWNER', 'DISPONENT'))
);

-- 2.5 Tarjeta
CREATE TABLE Tarjeta (
    Identificador_Tarjeta  INT          NOT NULL AUTO_INCREMENT,
    Tipo_Tarjeta           VARCHAR(45)  NOT NULL,
    Fecha_Emitida          DATE         NOT NULL,
    Codigo_Disposicion     INT          NOT NULL,
    CONSTRAINT pk_tarjeta PRIMARY KEY (Identificador_Tarjeta),
    CONSTRAINT uq_tarjeta_disposicion UNIQUE (Codigo_Disposicion),
    CONSTRAINT fk_tarjeta_disp FOREIGN KEY (Codigo_Disposicion) REFERENCES Disposicion (Codigo_Disposicion),
    CONSTRAINT ck_tarjeta_tipo CHECK (Tipo_Tarjeta IN ('classic', 'junior', 'gold'))
);

-- 2.6 Transacciones (+ Caracterizacion_Operacion, antes k_symbol)
CREATE TABLE Transacciones (
    Codigo_Transacciones       INT            NOT NULL AUTO_INCREMENT,
    Identificador_Cuenta       INT            NOT NULL,
    Fecha_Realizada            DATE           NOT NULL,
    Tipo_Movimiento            VARCHAR(45)    NOT NULL,
    Operacion_Efectuada        VARCHAR(30)    NULL,
    Caracterizacion_Operacion  VARCHAR(30)    NULL,
    Monto_Involucrado          DECIMAL(12,2)  NOT NULL,
    Saldo_Resultante_Cuenta    DECIMAL(12,2)  NOT NULL,
    Banco                      CHAR(2)        NULL,
    Cuenta_Contraparte         BIGINT         NULL,
    CONSTRAINT pk_transacciones PRIMARY KEY (Codigo_Transacciones),
    CONSTRAINT fk_trans_cuenta FOREIGN KEY (Identificador_Cuenta) REFERENCES Cuenta (Identificador_Cuenta),
    CONSTRAINT ck_trans_tipo  CHECK (Tipo_Movimiento IN ('PRIJEM', 'VYDAJ', 'VYBER')),
    CONSTRAINT ck_trans_monto CHECK (Monto_Involucrado >= 0),
    INDEX idx_trans_cuenta_fecha (Identificador_Cuenta, Fecha_Realizada, Codigo_Transacciones)
);

-- 2.7 Estado_Prestamo (código como letra, igual que en los datos)
CREATE TABLE Estado_Prestamo (
    Codigo_Estado       CHAR(1)      NOT NULL,
    Descripcion_Estado  VARCHAR(60)  NOT NULL,
    CONSTRAINT pk_estado_prestamo PRIMARY KEY (Codigo_Estado)
);

-- 2.8 Prestamos (FK a Cuenta, única: una cuenta tiene 0 o 1 préstamo)
CREATE TABLE Prestamos (
    Codigo_Prestamos       INT            NOT NULL AUTO_INCREMENT,
    Identificador_Cuenta   INT            NOT NULL,
    Fecha_Concedido        DATE           NOT NULL,
    Monto_Otorgado         DECIMAL(12,2)  NOT NULL,
    Duracion_Prestamo      INT            NOT NULL,
    Valor_Cuota_Periodica  DECIMAL(10,2)  NOT NULL,
    Codigo_Estado          CHAR(1)        NOT NULL,
    CONSTRAINT pk_prestamos PRIMARY KEY (Codigo_Prestamos),
    CONSTRAINT uq_prestamos_cuenta UNIQUE (Identificador_Cuenta),
    CONSTRAINT fk_prestamo_cuenta FOREIGN KEY (Identificador_Cuenta) REFERENCES Cuenta (Identificador_Cuenta),
    CONSTRAINT fk_prestamo_estado FOREIGN KEY (Codigo_Estado) REFERENCES Estado_Prestamo (Codigo_Estado),
    CONSTRAINT ck_prestamo_duracion CHECK (Duracion_Prestamo IN (12, 24, 36, 48, 60)),
    CONSTRAINT ck_prestamo_monto    CHECK (Monto_Otorgado > 0 AND Valor_Cuota_Periodica > 0)
);

-- 2.9 Ordenes_de_pago
CREATE TABLE Ordenes_de_pago (
    Codigo_Orden_Pago      INT            NOT NULL AUTO_INCREMENT,
    Identificador_Cuenta   INT            NOT NULL,
    Importe_Enviado        DECIMAL(10,2)  NOT NULL,
    Tipo_O_Finalidad_Pago  VARCHAR(30)    NULL,
    Cuenta_Destino         BIGINT         NOT NULL,
    Banco                  CHAR(2)        NOT NULL,
    CONSTRAINT pk_ordenes_pago PRIMARY KEY (Codigo_Orden_Pago),
    CONSTRAINT fk_orden_cuenta FOREIGN KEY (Identificador_Cuenta) REFERENCES Cuenta (Identificador_Cuenta),
    CONSTRAINT ck_orden_importe CHECK (Importe_Enviado > 0)
);

-- Tabla de auditoría (la llena el trigger trg_cliente_after_update)
CREATE TABLE Auditoria_Cliente (
    Id_Auditoria             INT          NOT NULL AUTO_INCREMENT,
    Identificador_Cliente    INT          NOT NULL,
    Fecha_Nacimiento_Antes   DATE         NULL,
    Fecha_Nacimiento_Despues DATE         NULL,
    Genero_Antes             CHAR(1)      NULL,
    Genero_Despues           CHAR(1)      NULL,
    Distrito_Antes           INT          NULL,
    Distrito_Despues         INT          NULL,
    Usuario                  VARCHAR(100) NOT NULL,
    Fecha_Cambio             DATETIME     NOT NULL DEFAULT CURRENT_TIMESTAMP,
    CONSTRAINT pk_auditoria_cliente PRIMARY KEY (Id_Auditoria)
);
"""

# Estas sentencias no tienen ';' internos, así que se pueden separar por ';'
with engine_srv.begin() as con:
    for sentencia in [s.strip() for s in DDL_MODELO.split(';') if s.strip()]:
        con.execute(text(sentencia))

engine = crear_engine(DB_MODELO)
consultar('SHOW TABLES')

OperationalError: (pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on 'localhost' ([WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión)")
(Background on this error at: https://sqlalche.me/e/21/e3q8)

### 4.1 Estructura de una tabla (ejemplo: `Transacciones`)

In [ ]:
consultar('DESCRIBE Transacciones')

<a id="5"></a>
## 5. Migración de los datos al modelo y validación

Cada tabla del modelo se llena con un `INSERT ... SELECT` desde el staging, aplicando las transformaciones del diccionario de datos:

| Tabla destino | Transformación principal |
|---|---|
| `Distrito` | `Cantidad_Municipios = A5+A6+A7+A8`; `Tasa_Desempleo = A13` (1996) |
| `Cliente` | `birth_number (AAMMDD)` → `Fecha_Nacimiento`; mes > 50 ⇒ `Genero = 'F'` y se le restan 50 al mes |
| `Estado_Prestamo` | Catálogo A–D con su descripción |
| `Transacciones`, `Ordenes_de_pago` | Textos vacíos (`''`, `' '`) → `NULL` |
| Resto | Copia directa con renombre de columnas |

Los triggers se crean **después** de la migración (sección 8). Si existieran antes, se dispararían para cada una de las 1,056,320 transacciones históricas.

*Tiempo aproximado: 1 minuto.*

In [ ]:
MIGRACION = [
    ('Distrito', f"""
        INSERT INTO Distrito (ID_Distrito, Nombre_Distrito, Region_que_pertenece, Cantidad_Municipios,
                              Tasa_Desempleo, Salario_Promedio, Cantidad_Emprendedores,
                              Grado_Urbanizacion, Numero_Habitantes)
        SELECT A1, A2, A3, A5 + A6 + A7 + A8, A13, A11, A14, A10, A4
        FROM {DB_ORIGEN}.district"""),

    ('Cliente', f"""
        INSERT INTO Cliente (Identificador_Cliente, Fecha_Nacimiento, Genero, ID_Distrito)
        SELECT client_id,
               CAST(CONCAT('19', LPAD(birth_number - IF(MOD(birth_number DIV 100, 100) > 50, 5000, 0), 6, '0')) AS DATE),
               IF(MOD(birth_number DIV 100, 100) > 50, 'F', 'M'),
               district_id
        FROM {DB_ORIGEN}.client"""),

    ('Cuenta', f"""
        INSERT INTO Cuenta (Identificador_Cuenta, Fecha_Apertura, Frecuencia_Emision_Estado_Cuenta, ID_Distrito)
        SELECT account_id, `date`, frequency, district_id
        FROM {DB_ORIGEN}.account"""),

    ('Disposicion', f"""
        INSERT INTO Disposicion (Codigo_Disposicion, Tipo_Disposicion, Identificador_Cliente, Identificador_Cuenta)
        SELECT disp_id, `type`, client_id, account_id
        FROM {DB_ORIGEN}.disp"""),

    ('Tarjeta', f"""
        INSERT INTO Tarjeta (Identificador_Tarjeta, Tipo_Tarjeta, Fecha_Emitida, Codigo_Disposicion)
        SELECT card_id, `type`, issued, disp_id
        FROM {DB_ORIGEN}.card"""),

    ('Estado_Prestamo', """
        INSERT INTO Estado_Prestamo (Codigo_Estado, Descripcion_Estado) VALUES
            ('A', 'Finalizado sin problemas'),
            ('B', 'Finalizado, préstamo no pagado'),
            ('C', 'Vigente sin problemas'),
            ('D', 'Vigente, cliente en deuda')"""),

    ('Prestamos', f"""
        INSERT INTO Prestamos (Codigo_Prestamos, Identificador_Cuenta, Fecha_Concedido, Monto_Otorgado,
                               Duracion_Prestamo, Valor_Cuota_Periodica, Codigo_Estado)
        SELECT loan_id, account_id, `date`, amount, duration, payments, status
        FROM {DB_ORIGEN}.loan"""),

    ('Ordenes_de_pago', f"""
        INSERT INTO Ordenes_de_pago (Codigo_Orden_Pago, Identificador_Cuenta, Importe_Enviado,
                                     Tipo_O_Finalidad_Pago, Cuenta_Destino, Banco)
        SELECT order_id, account_id, amount, NULLIF(TRIM(k_symbol), ''), account_to, bank_to
        FROM {DB_ORIGEN}.`order`"""),

    ('Transacciones', f"""
        INSERT INTO Transacciones (Codigo_Transacciones, Identificador_Cuenta, Fecha_Realizada, Tipo_Movimiento,
                                   Operacion_Efectuada, Caracterizacion_Operacion, Monto_Involucrado,
                                   Saldo_Resultante_Cuenta, Banco, Cuenta_Contraparte)
        SELECT trans_id, account_id, `date`, `type`,
               NULLIF(TRIM(operation), ''), NULLIF(TRIM(k_symbol), ''), amount,
               balance, NULLIF(TRIM(bank), ''), `account`
        FROM {DB_ORIGEN}.trans"""),
]

for tabla, sql in MIGRACION:
    inicio = time.time()
    filas = ejecutar(sql).rowcount
    print(f'{tabla:<16} {filas:>10,} filas  ({time.time() - inicio:.1f} s)')

### 5.1 Validación contra el diccionario de datos
Se comparan los conteos del modelo con los que reporta el diccionario de datos.

In [ ]:
VALIDACIONES = [
    ('Distritos',                      'SELECT COUNT(*) FROM Distrito', 77),
    ('Clientes',                       'SELECT COUNT(*) FROM Cliente', 5_369),
    ('Clientes hombres (M)',           "SELECT COUNT(*) FROM Cliente WHERE Genero = 'M'", 2_724),
    ('Clientes mujeres (F)',           "SELECT COUNT(*) FROM Cliente WHERE Genero = 'F'", 2_645),
    ('Cuentas',                        'SELECT COUNT(*) FROM Cuenta', 4_500),
    ('Disposiciones OWNER',            "SELECT COUNT(*) FROM Disposicion WHERE Tipo_Disposicion = 'OWNER'", 4_500),
    ('Disposiciones DISPONENT',        "SELECT COUNT(*) FROM Disposicion WHERE Tipo_Disposicion = 'DISPONENT'", 869),
    ('Tarjetas',                       'SELECT COUNT(*) FROM Tarjeta', 892),
    ('Préstamos',                      'SELECT COUNT(*) FROM Prestamos', 682),
    ('Órdenes de pago',                'SELECT COUNT(*) FROM Ordenes_de_pago', 6_471),
    ('Transacciones',                  'SELECT COUNT(*) FROM Transacciones', 1_056_320),
    ('Transacciones sin operación',    'SELECT COUNT(*) FROM Transacciones WHERE Operacion_Efectuada IS NULL', 183_114),
    ('Transacciones con k_symbol',     'SELECT COUNT(*) FROM Transacciones WHERE Caracterizacion_Operacion IS NOT NULL', 521_006),
]

with engine.connect() as con:
    resultado = pd.DataFrame(
        [(nombre, con.execute(text(sql)).scalar(), esperado) for nombre, sql, esperado in VALIDACIONES],
        columns=['Validación', 'En la BD', 'Diccionario'])
resultado['OK'] = resultado['En la BD'] == resultado['Diccionario']
resultado

Ejemplo del cliente 1, que en el diccionario aparece con nacimiento `1970-12-13` y género `F`:

In [ ]:
consultar("""
SELECT c.*, d.Nombre_Distrito
FROM Cliente c JOIN Distrito d ON d.ID_Distrito = c.ID_Distrito
WHERE c.Identificador_Cliente = 1
""")

<a id="6"></a>
## 6. Función `fn_saldo_cuenta`

**Qué hace:** devuelve el saldo de una cuenta en una fecha dada. Toma el `Saldo_Resultante_Cuenta` de la última transacción de la cuenta hasta esa fecha. Si la fecha es `NULL`, devuelve el saldo actual (último movimiento). Si la cuenta no tiene movimientos, devuelve 0.

**Dónde se usa:**
- En el trigger `trg_transacciones_before_insert`, para calcular el saldo de cada transacción nueva.
- En el procedimiento `sp_extracto_cuenta`, para el saldo inicial y final del periodo.
- En la pestaña "Saldo (función)" de la interfaz.

> Las rutinas se crean desde Python, así que no hace falta `DELIMITER` (eso solo lo necesita el cliente de consola o Workbench).

In [ ]:
FN_SALDO = """
CREATE FUNCTION fn_saldo_cuenta(p_cuenta INT, p_fecha DATE)
RETURNS DECIMAL(12,2)
READS SQL DATA
BEGIN
    DECLARE v_saldo DECIMAL(12,2);

    SET v_saldo = (
        SELECT Saldo_Resultante_Cuenta
        FROM Transacciones
        WHERE Identificador_Cuenta = p_cuenta
          AND (p_fecha IS NULL OR Fecha_Realizada <= p_fecha)
        ORDER BY Fecha_Realizada DESC, Codigo_Transacciones DESC
        LIMIT 1
    );

    RETURN IFNULL(v_saldo, 0);
END
"""
ejecutar('DROP FUNCTION IF EXISTS fn_saldo_cuenta')
ejecutar(FN_SALDO)
print('Función fn_saldo_cuenta creada.')

### 6.1 Prueba de la función

In [ ]:
consultar("""
SELECT 576                                     AS Cuenta,
       fn_saldo_cuenta(576, '1995-12-31')       AS Saldo_fin_1995,
       fn_saldo_cuenta(576, '1997-12-31')       AS Saldo_fin_1997,
       fn_saldo_cuenta(576, NULL)               AS Saldo_actual,
       fn_saldo_cuenta(576, '1990-01-01')       AS Saldo_antes_de_abrir
""")

<a id="7"></a>
## 7. Procedimiento `sp_extracto_cuenta`

**Qué hace:** genera el **extracto bancario** de una cuenta entre dos fechas.
1. Valida que la cuenta exista y que el rango de fechas sea correcto. Si no, lanza un error con `SIGNAL`.
2. Devuelve un **primer resultado** con el resumen: datos de la cuenta, saldo inicial, total de ingresos, total de egresos, número de movimientos y saldo final (usa `fn_saldo_cuenta`).
3. Devuelve un **segundo resultado** con el detalle de los movimientos del periodo.

In [ ]:
SP_EXTRACTO = """
CREATE PROCEDURE sp_extracto_cuenta(IN p_cuenta INT, IN p_desde DATE, IN p_hasta DATE)
BEGIN
    IF NOT EXISTS (SELECT 1 FROM Cuenta WHERE Identificador_Cuenta = p_cuenta) THEN
        SIGNAL SQLSTATE '45000' SET MESSAGE_TEXT = 'La cuenta indicada no existe';
    END IF;

    IF p_desde IS NULL OR p_hasta IS NULL OR p_desde > p_hasta THEN
        SIGNAL SQLSTATE '45000' SET MESSAGE_TEXT = 'Rango de fechas inválido (desde debe ser <= hasta)';
    END IF;

    -- 1) Resumen del periodo
    SELECT c.Identificador_Cuenta,
           d.Nombre_Distrito                                         AS Sucursal,
           c.Frecuencia_Emision_Estado_Cuenta                        AS Frecuencia,
           p_desde                                                   AS Desde,
           p_hasta                                                   AS Hasta,
           fn_saldo_cuenta(p_cuenta, DATE_SUB(p_desde, INTERVAL 1 DAY)) AS Saldo_Inicial,
           COALESCE(SUM(CASE WHEN t.Tipo_Movimiento = 'PRIJEM'  THEN t.Monto_Involucrado END), 0) AS Total_Ingresos,
           COALESCE(SUM(CASE WHEN t.Tipo_Movimiento <> 'PRIJEM' THEN t.Monto_Involucrado END), 0) AS Total_Egresos,
           COUNT(t.Codigo_Transacciones)                             AS Num_Movimientos,
           fn_saldo_cuenta(p_cuenta, p_hasta)                        AS Saldo_Final
    FROM Cuenta c
    JOIN Distrito d ON d.ID_Distrito = c.ID_Distrito
    LEFT JOIN Transacciones t
           ON t.Identificador_Cuenta = c.Identificador_Cuenta
          AND t.Fecha_Realizada BETWEEN p_desde AND p_hasta
    WHERE c.Identificador_Cuenta = p_cuenta
    GROUP BY c.Identificador_Cuenta, d.Nombre_Distrito, c.Frecuencia_Emision_Estado_Cuenta;

    -- 2) Detalle de movimientos
    SELECT Codigo_Transacciones, Fecha_Realizada, Tipo_Movimiento, Operacion_Efectuada,
           Caracterizacion_Operacion, Monto_Involucrado, Saldo_Resultante_Cuenta,
           Banco, Cuenta_Contraparte
    FROM Transacciones
    WHERE Identificador_Cuenta = p_cuenta
      AND Fecha_Realizada BETWEEN p_desde AND p_hasta
    ORDER BY Fecha_Realizada, Codigo_Transacciones;
END
"""
ejecutar('DROP PROCEDURE IF EXISTS sp_extracto_cuenta')
ejecutar(SP_EXTRACTO)
print('Procedimiento sp_extracto_cuenta creado.')

### 7.1 Llamar al procedimiento desde Python
El procedimiento devuelve **dos** conjuntos de resultados, así que se usa el cursor del driver (`callproc` + `nextset`).

In [ ]:
def texto_error(e):
    """Mensaje legible a partir de una excepción de MySQL."""
    orig = getattr(e, 'orig', e)
    args = getattr(orig, 'args', ())
    if len(args) >= 2:
        codigo, mensaje = args[0], args[1]
        explicacion = {
            1451: 'No se puede borrar: otros registros dependen de este (llave foránea).',
            1452: 'El valor de la llave foránea no existe (p. ej. la cuenta o el distrito).',
            3819: 'Se viola una restricción CHECK del modelo.',
            1048: 'Falta un campo obligatorio.',
        }.get(codigo, '')
        return f'Error {codigo}: {mensaje}' + (f' — {explicacion}' if explicacion else '')
    return str(orig)

def llamar_extracto(cuenta, desde, hasta):
    """Ejecuta sp_extracto_cuenta y devuelve (resumen, detalle) como DataFrames."""
    raw = engine.raw_connection()
    try:
        cur = raw.cursor()
        cur.callproc('sp_extracto_cuenta', (cuenta, desde, hasta))
        resumen = pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])
        cur.nextset()
        detalle = pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])
        cur.close()
        return resumen, detalle
    finally:
        raw.close()

resumen, detalle = llamar_extracto(576, date(1997, 1, 1), date(1997, 3, 31))
display(resumen)
display(detalle.head(10))

Prueba de la validación: una cuenta que no existe.

In [ ]:
try:
    llamar_extracto(999999, date(1997, 1, 1), date(1997, 3, 31))
except Exception as e:
    print(texto_error(e))

<a id="8"></a>
## 8. Triggers

### 8.1 `trg_transacciones_before_insert` (BEFORE INSERT en `Transacciones`)
Se ejecuta antes de guardar cada transacción nueva:
1. **Valida** que el monto sea mayor que 0. Si no, aborta con `SIGNAL`.
2. Si no se indica la fecha, usa la **fecha actual**.
3. **Valida** que la fecha no sea anterior a la apertura de la cuenta.
4. Si no se indica el saldo, lo **calcula automáticamente**: saldo previo de la cuenta (`fn_saldo_cuenta`) + monto si es ingreso (`PRIJEM`) o − monto si es egreso (`VYDAJ` / `VYBER`).

Gracias a este trigger, la interfaz no pide el saldo: lo calcula la base de datos.

In [ ]:
TRG_TRANS = """
CREATE TRIGGER trg_transacciones_before_insert
BEFORE INSERT ON Transacciones
FOR EACH ROW
BEGIN
    DECLARE v_apertura DATE;

    IF NEW.Monto_Involucrado IS NULL OR NEW.Monto_Involucrado <= 0 THEN
        SIGNAL SQLSTATE '45000' SET MESSAGE_TEXT = 'El monto de la transacción debe ser mayor que 0';
    END IF;

    IF NEW.Fecha_Realizada IS NULL THEN
        SET NEW.Fecha_Realizada = CURDATE();
    END IF;

    SELECT Fecha_Apertura INTO v_apertura
    FROM Cuenta WHERE Identificador_Cuenta = NEW.Identificador_Cuenta;

    IF v_apertura IS NOT NULL AND NEW.Fecha_Realizada < v_apertura THEN
        SIGNAL SQLSTATE '45000' SET MESSAGE_TEXT = 'La fecha es anterior a la apertura de la cuenta';
    END IF;

    IF NEW.Saldo_Resultante_Cuenta IS NULL THEN
        SET NEW.Saldo_Resultante_Cuenta =
            fn_saldo_cuenta(NEW.Identificador_Cuenta, NEW.Fecha_Realizada)
            + IF(NEW.Tipo_Movimiento = 'PRIJEM', NEW.Monto_Involucrado, -NEW.Monto_Involucrado);
    END IF;
END
"""
ejecutar('DROP TRIGGER IF EXISTS trg_transacciones_before_insert')
ejecutar(TRG_TRANS)
print('Trigger trg_transacciones_before_insert creado.')

### 8.2 `trg_cliente_after_update` (AFTER UPDATE en `Cliente`)
**Auditoría:** cada vez que se modifica un cliente, guarda en `Auditoria_Cliente` los valores anteriores y nuevos, el usuario de MySQL que hizo el cambio y la fecha y hora. Solo registra si algún dato cambió de verdad (se compara con `<=>`, que también funciona con `NULL`).

In [ ]:
TRG_CLIENTE = """
CREATE TRIGGER trg_cliente_after_update
AFTER UPDATE ON Cliente
FOR EACH ROW
BEGIN
    IF NOT (OLD.Fecha_Nacimiento <=> NEW.Fecha_Nacimiento)
       OR NOT (OLD.Genero <=> NEW.Genero)
       OR NOT (OLD.ID_Distrito <=> NEW.ID_Distrito) THEN
        INSERT INTO Auditoria_Cliente (Identificador_Cliente,
                                       Fecha_Nacimiento_Antes, Fecha_Nacimiento_Despues,
                                       Genero_Antes, Genero_Despues,
                                       Distrito_Antes, Distrito_Despues,
                                       Usuario)
        VALUES (NEW.Identificador_Cliente,
                OLD.Fecha_Nacimiento, NEW.Fecha_Nacimiento,
                OLD.Genero, NEW.Genero,
                OLD.ID_Distrito, NEW.ID_Distrito,
                CURRENT_USER());
    END IF;
END
"""
ejecutar('DROP TRIGGER IF EXISTS trg_cliente_after_update')
ejecutar(TRG_CLIENTE)
print('Trigger trg_cliente_after_update creado.')

### 8.3 Pruebas de los triggers
**Trigger 1:** se inserta un depósito de 1,500 en la cuenta 576 **sin indicar el saldo** y se verifica que el trigger lo calculó. Luego se intenta un monto negativo, que el trigger rechaza. Al final se borra la transacción de prueba.

In [ ]:
saldo_previo = ejecutar('SELECT fn_saldo_cuenta(576, NULL)').scalar()

r = ejecutar("""
    INSERT INTO Transacciones (Identificador_Cuenta, Fecha_Realizada, Tipo_Movimiento,
                               Operacion_Efectuada, Monto_Involucrado)
    VALUES (576, '1999-01-15', 'PRIJEM', 'VKLAD', 1500)
""")
id_prueba = r.lastrowid

print(f'Saldo previo: {saldo_previo:,.2f}  ->  esperado tras el depósito: {saldo_previo + 1500:,.2f}')
display(consultar('SELECT * FROM Transacciones WHERE Codigo_Transacciones = :id', {'id': id_prueba}))

try:
    ejecutar("""INSERT INTO Transacciones (Identificador_Cuenta, Tipo_Movimiento, Monto_Involucrado)
                VALUES (576, 'VYDAJ', -50)""")
except Exception as e:
    print('Rechazada por el trigger ->', texto_error(e))

ejecutar('DELETE FROM Transacciones WHERE Codigo_Transacciones = :id', {'id': id_prueba})
print('Transacción de prueba eliminada.')

**Trigger 2:** se cambia el distrito del cliente 1, se revisa la auditoría y se revierte el cambio (la reversión también queda auditada).

In [ ]:
distrito_original = ejecutar('SELECT ID_Distrito FROM Cliente WHERE Identificador_Cliente = 1').scalar()

ejecutar('UPDATE Cliente SET ID_Distrito = 1 WHERE Identificador_Cliente = 1')
ejecutar('UPDATE Cliente SET ID_Distrito = :d WHERE Identificador_Cliente = 1', {'d': distrito_original})

consultar('SELECT * FROM Auditoria_Cliente WHERE Identificador_Cliente = 1 ORDER BY Id_Auditoria')

<a id="9"></a>
## 9. Interfaz gráfica

La interfaz se construye con **ipywidgets** y funciona dentro del notebook (Jupyter, JupyterLab o VS Code). Tiene cinco pestañas:

| Pestaña | Operaciones |
|---|---|
| **Clientes** | Consultar (filtros por ID, distrito y género), Insertar, Actualizar (dispara la auditoría), Borrar |
| **Transacciones** | Consultar (por cuenta, rango de fechas y tipo), Insertar (el saldo lo calcula el trigger), Actualizar, Borrar |
| **Extracto (procedimiento)** | Ejecuta `sp_extracto_cuenta` y muestra el resumen y el detalle |
| **Saldo (función)** | Ejecuta `fn_saldo_cuenta` para una cuenta y fecha |
| **Auditoría (trigger)** | Muestra lo que registró `trg_cliente_after_update` |

Todas las sentencias usan **parámetros** (`:nombre`), nunca concatenación de texto, para evitar inyección SQL. Los errores de MySQL (llaves foráneas, `CHECK`, `SIGNAL` de los triggers) se muestran como mensajes en la interfaz.

### 9.1 Componentes comunes

In [ ]:
import ipywidgets as w
from IPython.display import display, clear_output, HTML

ESTILO = {'description_width': '170px'}
ANCHO = w.Layout(width='460px')

def campo(clase, descripcion, **kw):
    return clase(description=descripcion, style=ESTILO, layout=ANCHO, **kw)

def boton(descripcion, icono, estilo):
    return w.Button(description=descripcion, icon=icono, button_style=estilo, layout=w.Layout(width='180px'))

def mensaje(salida, texto, tipo='ok', df=None):
    color = {'ok': '#2e7d32', 'error': '#c62828', 'info': '#1565c0'}[tipo]
    with salida:
        clear_output()
        display(HTML(f"<div style='margin:6px 0;padding:6px 10px;border-left:4px solid {color}'>{texto}</div>"))
        if df is not None:
            display(df)

def seccion(titulo, ayuda, *hijos):
    return w.VBox([w.HTML(f'<h4 style="margin:4px 0">{titulo}</h4><small>{ayuda}</small>'), *hijos],
                  layout=w.Layout(padding='8px'))

def pestanas(pares):
    tab = w.Tab(children=[contenido for _, contenido in pares])
    for i, (titulo, _) in enumerate(pares):
        tab.set_title(i, titulo)
    return tab

def a_none(valor):
    """Convierte 0 / '' en None para los filtros y campos opcionales."""
    return None if valor in (0, '', None) else valor

# Catálogos para las listas desplegables
OPC_DISTRITO = [(f'{r.Nombre_Distrito} ({r.ID_Distrito})', int(r.ID_Distrito))
                for r in consultar('SELECT ID_Distrito, Nombre_Distrito FROM Distrito ORDER BY Nombre_Distrito').itertuples()]
OPC_GENERO = [('Masculino (M)', 'M'), ('Femenino (F)', 'F')]
OPC_TIPO_MOV = [('PRIJEM - ingreso', 'PRIJEM'), ('VYDAJ - egreso', 'VYDAJ'), ('VYBER - retiro en efectivo', 'VYBER')]
OPC_OPERACION = [('(ninguna)', None),
                 ('VKLAD - depósito en efectivo', 'VKLAD'),
                 ('VYBER - retiro en efectivo', 'VYBER'),
                 ('VYBER KARTOU - retiro con tarjeta', 'VYBER KARTOU'),
                 ('PREVOD Z UCTU - cobro desde otro banco', 'PREVOD Z UCTU'),
                 ('PREVOD NA UCET - transferencia a otro banco', 'PREVOD NA UCET')]
OPC_CARACTERIZACION = [('(ninguna)', None)] + [
    (v, v) for v in consultar("""SELECT DISTINCT Caracterizacion_Operacion AS v FROM Transacciones
                                 WHERE Caracterizacion_Operacion IS NOT NULL ORDER BY 1""")['v']]
OPC_BANCO = [('(ninguno)', None)] + [(b, b) for b in 'AB CD EF GH IJ KL MN OP QR ST UV WX YZ'.split()]
print('Componentes listos.')

### 9.2 Pestaña Clientes (CRUD sobre `Cliente`)

In [ ]:
SQL_CLIENTE_DETALLE = """
    SELECT c.Identificador_Cliente, c.Fecha_Nacimiento, c.Genero, c.ID_Distrito, d.Nombre_Distrito,
           (SELECT COUNT(*) FROM Disposicion x WHERE x.Identificador_Cliente = c.Identificador_Cliente) AS Num_Cuentas
    FROM Cliente c JOIN Distrito d ON d.ID_Distrito = c.ID_Distrito
"""

# ---- Consultar ----
cq_id   = campo(w.IntText, 'ID cliente (0 = todos)', value=0)
cq_dist = campo(w.Dropdown, 'Distrito', options=[('(todos)', None)] + OPC_DISTRITO, value=None)
cq_gen  = campo(w.Dropdown, 'Género', options=[('(todos)', None)] + OPC_GENERO, value=None)
cq_lim  = campo(w.BoundedIntText, 'Máximo de filas', value=25, min=1, max=10000)
cq_btn, cq_out = boton('Consultar', 'search', 'info'), w.Output()

def cliente_consultar(_=None):
    filtros, params = [], {'lim': cq_lim.value}
    if a_none(cq_id.value):
        filtros.append('c.Identificador_Cliente = :id'); params['id'] = cq_id.value
    if cq_dist.value is not None:
        filtros.append('c.ID_Distrito = :dist'); params['dist'] = cq_dist.value
    if cq_gen.value is not None:
        filtros.append('c.Genero = :gen'); params['gen'] = cq_gen.value
    where = (' WHERE ' + ' AND '.join(filtros)) if filtros else ''
    try:
        df = consultar(SQL_CLIENTE_DETALLE + where + ' ORDER BY c.Identificador_Cliente LIMIT :lim', params)
        mensaje(cq_out, f'{len(df)} cliente(s) encontrados.', 'info', df)
    except Exception as e:
        mensaje(cq_out, texto_error(e), 'error')
cq_btn.on_click(cliente_consultar)

# ---- Insertar ----
ci_fecha = campo(w.DatePicker, 'Fecha de nacimiento', value=date(1980, 1, 1))
ci_gen   = campo(w.Dropdown, 'Género', options=OPC_GENERO)
ci_dist  = campo(w.Dropdown, 'Distrito', options=OPC_DISTRITO)
ci_btn, ci_out = boton('Insertar cliente', 'plus', 'success'), w.Output()

def cliente_insertar(_=None):
    if ci_fecha.value is None:
        return mensaje(ci_out, 'Seleccione la fecha de nacimiento.', 'error')
    try:
        nuevo = ejecutar('INSERT INTO Cliente (Fecha_Nacimiento, Genero, ID_Distrito) VALUES (:f, :g, :d)',
                         {'f': ci_fecha.value, 'g': ci_gen.value, 'd': ci_dist.value}).lastrowid
        df = consultar(SQL_CLIENTE_DETALLE + ' WHERE c.Identificador_Cliente = :id', {'id': nuevo})
        mensaje(ci_out, f'Cliente insertado con ID <b>{nuevo}</b>.', 'ok', df)
    except Exception as e:
        mensaje(ci_out, texto_error(e), 'error')
ci_btn.on_click(cliente_insertar)

# ---- Actualizar ----
cu_id     = campo(w.IntText, 'ID cliente', value=1)
cu_cargar = boton('Cargar', 'download', 'info')
cu_fecha  = campo(w.DatePicker, 'Fecha de nacimiento', disabled=True)
cu_gen    = campo(w.Dropdown, 'Género', options=OPC_GENERO, disabled=True)
cu_dist   = campo(w.Dropdown, 'Distrito', options=OPC_DISTRITO, disabled=True)
cu_btn, cu_out = boton('Guardar cambios', 'save', 'warning'), w.Output()
cu_btn.disabled = True
cu_estado = {'id': None}

def cliente_cargar(_=None):
    df = consultar('SELECT * FROM Cliente WHERE Identificador_Cliente = :id', {'id': cu_id.value})
    if df.empty:
        cu_estado['id'] = None
        for x in (cu_fecha, cu_gen, cu_dist, cu_btn): x.disabled = True
        return mensaje(cu_out, f'No existe el cliente {cu_id.value}.', 'error')
    fila = df.iloc[0]
    cu_estado['id'] = int(fila.Identificador_Cliente)
    cu_fecha.value, cu_gen.value, cu_dist.value = fila.Fecha_Nacimiento, fila.Genero, int(fila.ID_Distrito)
    for x in (cu_fecha, cu_gen, cu_dist, cu_btn): x.disabled = False
    mensaje(cu_out, f'Cliente {cu_estado["id"]} cargado. Modifique los campos y pulse "Guardar cambios".', 'info')

def cliente_actualizar(_=None):
    if cu_estado['id'] is None:
        return mensaje(cu_out, 'Primero cargue un cliente.', 'error')
    try:
        n = ejecutar("""UPDATE Cliente SET Fecha_Nacimiento = :f, Genero = :g, ID_Distrito = :d
                        WHERE Identificador_Cliente = :id""",
                     {'f': cu_fecha.value, 'g': cu_gen.value, 'd': cu_dist.value, 'id': cu_estado['id']}).rowcount
        aud = consultar("""SELECT * FROM Auditoria_Cliente WHERE Identificador_Cliente = :id
                           ORDER BY Id_Auditoria DESC LIMIT 5""", {'id': cu_estado['id']})
        texto = ('Cliente actualizado. Últimos registros de auditoría (trigger):' if n
                 else 'No hubo cambios que guardar.')
        mensaje(cu_out, texto, 'ok' if n else 'info', aud)
    except Exception as e:
        mensaje(cu_out, texto_error(e), 'error')
cu_cargar.on_click(cliente_cargar)
cu_btn.on_click(cliente_actualizar)

# ---- Borrar ----
cb_id   = campo(w.IntText, 'ID cliente', value=0)
cb_conf = w.Checkbox(description='Confirmo que deseo borrar este cliente', indent=False)
cb_btn, cb_out = boton('Borrar cliente', 'trash', 'danger'), w.Output()

def cliente_borrar(_=None):
    if not cb_conf.value:
        return mensaje(cb_out, 'Marque la casilla de confirmación.', 'error')
    try:
        n = ejecutar('DELETE FROM Cliente WHERE Identificador_Cliente = :id', {'id': cb_id.value}).rowcount
        mensaje(cb_out, f'Cliente {cb_id.value} eliminado.' if n else f'No existe el cliente {cb_id.value}.',
                'ok' if n else 'error')
    except Exception as e:
        mensaje(cb_out, texto_error(e), 'error')
    cb_conf.value = False
cb_btn.on_click(cliente_borrar)

panel_clientes = pestanas([
    ('Consultar', seccion('Consultar clientes', 'Filtros opcionales.', cq_id, cq_dist, cq_gen, cq_lim, cq_btn, cq_out)),
    ('Insertar', seccion('Nuevo cliente', 'El ID lo asigna la base de datos (AUTO_INCREMENT).', ci_fecha, ci_gen, ci_dist, ci_btn, ci_out)),
    ('Actualizar', seccion('Modificar cliente', 'Cada cambio queda registrado por el trigger de auditoría.',
                           w.HBox([cu_id, cu_cargar]), cu_fecha, cu_gen, cu_dist, cu_btn, cu_out)),
    ('Borrar', seccion('Eliminar cliente',
                       'Un cliente con cuentas (disposiciones) no se puede borrar: lo impide la llave foránea.',
                       cb_id, cb_conf, cb_btn, cb_out)),
])
print('Panel de clientes listo.')

### 9.3 Pestaña Transacciones (CRUD sobre `Transacciones`)

In [ ]:
# ---- Consultar ----
tq_cuenta = campo(w.IntText, 'Cuenta (0 = todas)', value=576)
tq_desde  = campo(w.DatePicker, 'Desde (opcional)')
tq_hasta  = campo(w.DatePicker, 'Hasta (opcional)')
tq_tipo   = campo(w.Dropdown, 'Tipo de movimiento', options=[('(todos)', None)] + OPC_TIPO_MOV, value=None)
tq_lim    = campo(w.BoundedIntText, 'Máximo de filas', value=25, min=1, max=10000)
tq_btn, tq_out = boton('Consultar', 'search', 'info'), w.Output()

def trans_consultar(_=None):
    filtros, params = [], {'lim': tq_lim.value}
    if a_none(tq_cuenta.value):
        filtros.append('Identificador_Cuenta = :c'); params['c'] = tq_cuenta.value
    if tq_desde.value:
        filtros.append('Fecha_Realizada >= :desde'); params['desde'] = tq_desde.value
    if tq_hasta.value:
        filtros.append('Fecha_Realizada <= :hasta'); params['hasta'] = tq_hasta.value
    if tq_tipo.value:
        filtros.append('Tipo_Movimiento = :tipo'); params['tipo'] = tq_tipo.value
    where = (' WHERE ' + ' AND '.join(filtros)) if filtros else ''
    try:
        df = consultar('SELECT * FROM Transacciones' + where +
                       ' ORDER BY Fecha_Realizada DESC, Codigo_Transacciones DESC LIMIT :lim', params)
        mensaje(tq_out, f'{len(df)} transacción(es) (más recientes primero).', 'info', df)
    except Exception as e:
        mensaje(tq_out, texto_error(e), 'error')
tq_btn.on_click(trans_consultar)

# ---- Formulario reutilizable (insertar / actualizar) ----
def formulario_transaccion():
    return {
        'Fecha_Realizada':           campo(w.DatePicker, 'Fecha', value=date.today()),
        'Tipo_Movimiento':           campo(w.Dropdown, 'Tipo de movimiento', options=OPC_TIPO_MOV),
        'Operacion_Efectuada':       campo(w.Dropdown, 'Operación', options=OPC_OPERACION, value=None),
        'Caracterizacion_Operacion': campo(w.Dropdown, 'Caracterización (k_symbol)', options=OPC_CARACTERIZACION, value=None),
        'Monto_Involucrado':         campo(w.BoundedFloatText, 'Monto (Kč)', value=1000, min=-1e9, max=1e9),
        'Banco':                     campo(w.Dropdown, 'Banco contraparte', options=OPC_BANCO, value=None),
        'Cuenta_Contraparte':        campo(w.IntText, 'Cuenta contraparte (0 = ninguna)', value=0),
    }

def valores(form):
    v = {k: x.value for k, x in form.items()}
    v['Cuenta_Contraparte'] = a_none(v['Cuenta_Contraparte'])
    return v

# ---- Insertar ----
ti_cuenta = campo(w.IntText, 'Cuenta', value=576)
ti_form = formulario_transaccion()
ti_btn, ti_out = boton('Insertar transacción', 'plus', 'success'), w.Output()

def trans_insertar(_=None):
    params = valores(ti_form) | {'c': ti_cuenta.value}
    try:
        nuevo = ejecutar("""
            INSERT INTO Transacciones (Identificador_Cuenta, Fecha_Realizada, Tipo_Movimiento, Operacion_Efectuada,
                                       Caracterizacion_Operacion, Monto_Involucrado, Banco, Cuenta_Contraparte)
            VALUES (:c, :Fecha_Realizada, :Tipo_Movimiento, :Operacion_Efectuada,
                    :Caracterizacion_Operacion, :Monto_Involucrado, :Banco, :Cuenta_Contraparte)""", params).lastrowid
        df = consultar('SELECT * FROM Transacciones WHERE Codigo_Transacciones = :id', {'id': nuevo})
        mensaje(ti_out, f'Transacción <b>{nuevo}</b> registrada. Saldo resultante calculado por el trigger: '
                        f'<b>{df.Saldo_Resultante_Cuenta[0]:,.2f} Kč</b>', 'ok', df)
    except Exception as e:
        mensaje(ti_out, texto_error(e), 'error')
ti_btn.on_click(trans_insertar)

# ---- Actualizar ----
tu_id     = campo(w.IntText, 'Código transacción', value=0)
tu_cargar = boton('Cargar', 'download', 'info')
tu_form   = formulario_transaccion()
tu_saldo  = campo(w.FloatText, 'Saldo resultante (Kč)', value=0)
tu_btn, tu_out = boton('Guardar cambios', 'save', 'warning'), w.Output()
tu_estado = {'id': None}

def habilitar_tu(si):
    for x in [*tu_form.values(), tu_saldo, tu_btn]:
        x.disabled = not si
habilitar_tu(False)

def trans_cargar(_=None):
    df = consultar('SELECT * FROM Transacciones WHERE Codigo_Transacciones = :id', {'id': tu_id.value})
    if df.empty:
        tu_estado['id'] = None; habilitar_tu(False)
        return mensaje(tu_out, f'No existe la transacción {tu_id.value}.', 'error')
    fila = df.iloc[0]
    tu_estado['id'] = int(fila.Codigo_Transacciones)
    for k, x in tu_form.items():
        valor = fila[k]
        if pd.isna(valor):
            valor = 0 if k == 'Cuenta_Contraparte' else None
        elif k in ('Monto_Involucrado',):
            valor = float(valor)
        elif k == 'Cuenta_Contraparte':
            valor = int(valor)
        x.value = valor
    tu_saldo.value = float(fila.Saldo_Resultante_Cuenta)
    habilitar_tu(True)
    mensaje(tu_out, f'Transacción {tu_estado["id"]} (cuenta {int(fila.Identificador_Cuenta)}) cargada.', 'info')

def trans_actualizar(_=None):
    if tu_estado['id'] is None:
        return mensaje(tu_out, 'Primero cargue una transacción.', 'error')
    params = valores(tu_form) | {'saldo': tu_saldo.value, 'id': tu_estado['id']}
    try:
        ejecutar("""
            UPDATE Transacciones
            SET Fecha_Realizada = :Fecha_Realizada, Tipo_Movimiento = :Tipo_Movimiento,
                Operacion_Efectuada = :Operacion_Efectuada, Caracterizacion_Operacion = :Caracterizacion_Operacion,
                Monto_Involucrado = :Monto_Involucrado, Saldo_Resultante_Cuenta = :saldo,
                Banco = :Banco, Cuenta_Contraparte = :Cuenta_Contraparte
            WHERE Codigo_Transacciones = :id""", params)
        df = consultar('SELECT * FROM Transacciones WHERE Codigo_Transacciones = :id', {'id': tu_estado['id']})
        mensaje(tu_out, 'Transacción actualizada.', 'ok', df)
    except Exception as e:
        mensaje(tu_out, texto_error(e), 'error')
tu_cargar.on_click(trans_cargar)
tu_btn.on_click(trans_actualizar)

# ---- Borrar ----
tb_id   = campo(w.IntText, 'Código transacción', value=0)
tb_conf = w.Checkbox(description='Confirmo que deseo borrar esta transacción', indent=False)
tb_btn, tb_out = boton('Borrar transacción', 'trash', 'danger'), w.Output()

def trans_borrar(_=None):
    if not tb_conf.value:
        return mensaje(tb_out, 'Marque la casilla de confirmación.', 'error')
    try:
        n = ejecutar('DELETE FROM Transacciones WHERE Codigo_Transacciones = :id', {'id': tb_id.value}).rowcount
        mensaje(tb_out, f'Transacción {tb_id.value} eliminada.' if n else f'No existe la transacción {tb_id.value}.',
                'ok' if n else 'error')
    except Exception as e:
        mensaje(tb_out, texto_error(e), 'error')
    tb_conf.value = False
tb_btn.on_click(trans_borrar)

panel_transacciones = pestanas([
    ('Consultar', seccion('Consultar transacciones', 'Filtros opcionales.',
                          tq_cuenta, tq_desde, tq_hasta, tq_tipo, tq_lim, tq_btn, tq_out)),
    ('Insertar', seccion('Nueva transacción',
                         'No se pide el saldo: lo calcula el trigger <code>trg_transacciones_before_insert</code>, '
                         'que además rechaza montos ≤ 0 y fechas anteriores a la apertura de la cuenta.',
                         ti_cuenta, *ti_form.values(), ti_btn, ti_out)),
    ('Actualizar', seccion('Modificar transacción',
                           'Corrige datos de un movimiento. El saldo se edita manualmente '
                           '(el trigger solo actúa al insertar). La restricción CHECK impide montos negativos.',
                           w.HBox([tu_id, tu_cargar]), *tu_form.values(), tu_saldo, tu_btn, tu_out)),
    ('Borrar', seccion('Eliminar transacción', '', tb_id, tb_conf, tb_btn, tb_out)),
])
print('Panel de transacciones listo.')

### 9.4 Pestañas de procedimiento, función y auditoría

In [ ]:
# ---- Procedimiento ----
pe_cuenta = campo(w.IntText, 'Cuenta', value=576)
pe_desde  = campo(w.DatePicker, 'Desde', value=date(1997, 1, 1))
pe_hasta  = campo(w.DatePicker, 'Hasta', value=date(1997, 12, 31))
pe_btn, pe_out = boton('Generar extracto', 'file-text', 'primary'), w.Output()

def extracto(_=None):
    try:
        resumen, detalle = llamar_extracto(pe_cuenta.value, pe_desde.value, pe_hasta.value)
        mensaje(pe_out, 'Resumen del periodo:', 'info', resumen)
        with pe_out:
            display(HTML(f'<b>Detalle ({len(detalle)} movimientos):</b>'))
            display(detalle)
    except Exception as e:
        mensaje(pe_out, texto_error(e), 'error')
pe_btn.on_click(extracto)

# ---- Función ----
fs_cuenta = campo(w.IntText, 'Cuenta', value=576)
fs_fecha  = campo(w.DatePicker, 'A la fecha (vacío = actual)')
fs_btn, fs_out = boton('Consultar saldo', 'money', 'primary'), w.Output()

def saldo(_=None):
    try:
        valor = ejecutar('SELECT fn_saldo_cuenta(:c, :f)', {'c': fs_cuenta.value, 'f': fs_fecha.value}).scalar()
        cuando = fs_fecha.value.isoformat() if fs_fecha.value else 'hoy (último movimiento)'
        mensaje(fs_out, f'Saldo de la cuenta <b>{fs_cuenta.value}</b> a {cuando}: <b>{valor:,.2f} Kč</b>')
    except Exception as e:
        mensaje(fs_out, texto_error(e), 'error')
fs_btn.on_click(saldo)

# ---- Auditoría ----
au_btn, au_out = boton('Refrescar', 'refresh', 'info'), w.Output()
def auditoria(_=None):
    df = consultar('SELECT * FROM Auditoria_Cliente ORDER BY Id_Auditoria DESC LIMIT 100')
    mensaje(au_out, f'{len(df)} registro(s) de auditoría (más recientes primero).', 'info', df)
au_btn.on_click(auditoria)

panel_extracto = seccion('Extracto bancario', 'Llama a <code>CALL sp_extracto_cuenta(cuenta, desde, hasta)</code>.',
                         pe_cuenta, pe_desde, pe_hasta, pe_btn, pe_out)
panel_saldo = seccion('Saldo de una cuenta', 'Llama a <code>SELECT fn_saldo_cuenta(cuenta, fecha)</code>.',
                      fs_cuenta, fs_fecha, fs_btn, fs_out)
panel_auditoria = seccion('Auditoría de clientes', 'Registros que inserta <code>trg_cliente_after_update</code>.',
                          au_btn, au_out)
print('Paneles de rutinas listos.')

### 9.5 Lanzar la interfaz
Ejecuta la celda y usa las pestañas. Algunas pruebas sugeridas:
- **Clientes → Insertar** un cliente nuevo, luego **Actualizar** su distrito y mirar la pestaña **Auditoría**. Al final, **Borrar** ese cliente (se puede porque no tiene cuentas).
- **Clientes → Borrar** el cliente 1: la llave foránea lo impide porque tiene cuentas.
- **Transacciones → Insertar** un retiro (`VYDAJ`) en la cuenta 576 y ver el saldo calculado. Probar un monto negativo para ver el rechazo del trigger.

In [ ]:
app = pestanas([
    ('Clientes', panel_clientes),
    ('Transacciones', panel_transacciones),
    ('Extracto (procedimiento)', panel_extracto),
    ('Saldo (función)', panel_saldo),
    ('Auditoría (trigger)', panel_auditoria),
])
display(w.VBox([
    w.HTML('<h2 style="margin:0">Banco Berka — Gestión de clientes y operaciones</h2>'
           f'<small>Base de datos: <code>{DB_MODELO}</code> en {MYSQL_HOST}:{MYSQL_PORT}</small>'),
    app,
]))

<a id="10"></a>
## 10. Resumen de objetos creados
Consulta al diccionario del motor (`information_schema`) para listar lo implementado en esta entrega.

In [ ]:
display(consultar("""
    SELECT TABLE_NAME AS Tabla, TABLE_ROWS AS Filas_aprox
    FROM information_schema.TABLES
    WHERE TABLE_SCHEMA = DATABASE() ORDER BY TABLE_NAME"""))

display(consultar("""
    SELECT TABLE_NAME AS Tabla, CONSTRAINT_NAME AS Restriccion, CONSTRAINT_TYPE AS Tipo
    FROM information_schema.TABLE_CONSTRAINTS
    WHERE TABLE_SCHEMA = DATABASE() ORDER BY TABLE_NAME, CONSTRAINT_TYPE"""))

display(consultar("""
    SELECT TRIGGER_NAME AS `Trigger`, ACTION_TIMING AS Momento, EVENT_MANIPULATION AS Evento,
           EVENT_OBJECT_TABLE AS Tabla
    FROM information_schema.TRIGGERS WHERE TRIGGER_SCHEMA = DATABASE()"""))

display(consultar("""
    SELECT ROUTINE_NAME AS Rutina, ROUTINE_TYPE AS Tipo, DTD_IDENTIFIER AS Retorna
    FROM information_schema.ROUTINES WHERE ROUTINE_SCHEMA = DATABASE()"""))